# 5. What do the models use?

**Question.** Which inputs drive the predictions, in which direction, and is the story the same for
logistic regression and XGBoost? Importance tells us what the model uses, **not** what causes disease.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nhanes import nb, plots
from nhanes.features import CATEGORICAL, FEATURES, LABELS, NUMERIC, TARGET

ctx = nb.context()
df, R = ctx.df, ctx.R
pct = nb.pct
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 160)
print("mode:", "synthetic fixture (CI)" if ctx.fixture else "real NHANES data")
print(f"{len(df):,} adults, {int(df[TARGET].sum()):,} reporting heart disease")

mode: real NHANES data
22,512 adults, 1,637 reporting heart disease


## 5.1 Permutation importance and SHAP

In [2]:
from nhanes.explain import shap_for_xgboost

models = ctx.models()
test = df[df.split == "test"].reset_index(drop=True)
sh = shap_for_xgboost(models["xgboost"], test)
nb.check(
    "top mean |SHAP| value",
    sh["table"][0]["mean_abs_shap"],
    R["importance"]["shap_xgboost_mean_abs"][0]["mean_abs_shap"],
)
nb.show(plots.plot_importance(R["importance"]["permutation_selected"], sh["table"]))
nb.show(plots.plot_shap_beeswarm(sh))

C:\Users\vansh\Desktop\Projects\nhanes-heart-risk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Check: top mean |SHAP| value: notebook 1.07623 vs results.json 1.07623 (matches)


<Figure size 1320x528 with 2 Axes>

<Figure size 935x572 with 2 Axes>

## 5.2 Logistic regression: odds ratios

In [3]:
orr = pd.DataFrame(R["odds_ratios_logistic"])
display(orr.head(10))

,feature,coefficient,odds_ratio
0,age_years,1.210,3.354
1,smoking_status_current,0.541,1.717
2,ever_heavy_drinker_yes,0.377,1.458
3,total_chol,-0.318,0.728
4,diabetes_no,-0.272,0.762
5,smoking_status_never,-0.225,0.798
6,diabetes_yes,0.223,1.250
7,diabetes_borderline,0.135,1.145
8,drinks_per_day,-0.132,0.876
9,hdl,-0.114,0.892


## 5.3 Does the model's risk track observed risk across age?

In [4]:
sel = R["selection"]["selected"]
pr = ctx.pred
tt = pr[pr.split == "test"].copy()
tt["age_group"] = pd.cut(
    tt.age_years, [19, 29, 39, 49, 59, 69, 80], labels=["20s", "30s", "40s", "50s", "60s", "70-80"]
)
byage = tt.groupby("age_group", observed=True).agg(
    observed=(TARGET, "mean"), predicted=(f"p_{sel}", "mean"), adults=(TARGET, "size")
)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(
    byage.index.astype(str),
    byage.observed * 100,
    marker="o",
    color="#0072B2",
    label="observed rate",
)
ax.plot(
    byage.index.astype(str),
    byage.predicted * 100,
    marker="s",
    ls="--",
    color="#D55E00",
    label="mean predicted risk",
)
ax.set(ylabel="%", title="Test cycle, by age group")
ax.legend(frameon=False)
nb.show(fig)
display(byage)

<Figure size 715x440 with 1 Axes>

,observed,predicted,adults
age_group,,,
20s,0.007,0.006,827
30s,0.013,0.013,857
40s,0.017,0.027,810
50s,0.068,0.054,913
60s,0.131,0.117,1096
70-80,0.213,0.225,1031


## 5.4 SHAP dependence: total cholesterol

In [5]:
names = sh["names"]
j = names.index("total_chol")
fig, ax = plt.subplots(figsize=(6.5, 4.2))
older = (test.age_years >= 60).to_numpy()
ax.scatter(
    test.total_chol[~older],
    sh["values"][~older, j],
    s=6,
    alpha=0.5,
    color="#0072B2",
    label="under 60",
)
ax.scatter(
    test.total_chol[older],
    sh["values"][older, j],
    s=6,
    alpha=0.5,
    color="#D55E00",
    label="60 and over",
)
ax.axhline(0, color="black", lw=0.7)
ax.set(
    xlabel="Total cholesterol (mg/dL), missing values omitted",
    ylabel="SHAP value (log-odds)",
    title="XGBoost: effect of total cholesterol",
)
ax.legend(frameon=False)
nb.show(fig)

<Figure size 715x462 with 1 Axes>

## 5.5 Take features away and refit logistic regression

Does the cholesterol finding come from a treatment effect, and does the model still work without lab
tests (which a person may not have at hand)? Same data, same tuned C, only the inputs change.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from nhanes.features import make_preprocessor

C = R["models"]["logistic_regression"]["params"]["C"]
tr, te = df[df.split == "train"], df[df.split == "test"]
labs = ["total_chol", "hdl", "hba1c", "fasting_glucose"]
variants = {
    "all features": (NUMERIC, CATEGORICAL),
    "without total cholesterol": ([c for c in NUMERIC if c != "total_chol"], CATEGORICAL),
    "without any blood test": ([c for c in NUMERIC if c not in labs], CATEGORICAL),
    "age and sex only": (["age_years"], ["sex"]),
}
rows = []
for name, (num, cat) in variants.items():
    pipe = Pipeline(
        [
            ("pre", make_preprocessor("median_indicator", True, num, cat)),
            ("clf", LogisticRegression(C=C, max_iter=5000)),
        ]
    )
    pipe.fit(tr[FEATURES], tr[TARGET])
    rows.append(
        {
            "inputs": name,
            "test ROC-AUC": roc_auc_score(te[TARGET], pipe.predict_proba(te[FEATURES])[:, 1]),
        }
    )
abl = pd.DataFrame(rows).set_index("inputs")
display(abl)
nb.check(
    "all-features LR test ROC-AUC",
    abl.loc["all features", "test ROC-AUC"],
    R["models"]["logistic_regression"]["test"]["roc_auc"],
)

,test ROC-AUC
inputs,
all features,0.829
without total cholesterol,0.822
without any blood test,0.821
age and sex only,0.795


Check: all-features LR test ROC-AUC: notebook 0.828528 vs results.json 0.828528 (matches)


## 5.6 Do the two models agree on what matters?

In [7]:
from scipy.stats import spearmanr

a = pd.DataFrame(R["importance"]["permutation_selected"]).set_index("feature").auc_drop_mean
b = (
    pd.DataFrame(R["importance"]["permutation_xgboost"])
    .set_index("feature")
    .auc_drop_mean.reindex(a.index)
)
rho = spearmanr(a, b)[0]
print(
    f"Spearman rank correlation of permutation importances, {R['selection']['selected']} vs XGBoost: {rho:.2f}"
)

Spearman rank correlation of permutation importances, logistic_regression vs XGBoost: 0.75


## What this means

In [8]:
top = R["importance"]["permutation_selected"]
first, second = top[0], top[1]
chol_or = orr.loc[orr.feature == "total_chol", "odds_ratio"]
nb.say(
    f"- **{LABELS.get(first['feature'], first['feature'])}** is the most used input: shuffling it lowers test ROC-AUC by "
    f"{first['auc_drop_mean'] * 100:.1f} points, against {second['auc_drop_mean'] * 100:.1f} for the next, "
    f"{LABELS.get(second['feature'], second['feature'])}.\n"
    f"- Using only age and sex gives {pct(abl.loc['age and sex only', 'test ROC-AUC'])}; all features give {pct(abl.loc['all features', 'test ROC-AUC'])}; "
    f"dropping every blood test gives {pct(abl.loc['without any blood test', 'test ROC-AUC'])}.\n"
    f"- The two models rank features similarly (Spearman {rho:.2f}).\n"
    + (
        f"- The logistic odds ratio for total cholesterol is {chol_or.iloc[0]:.2f} per standard deviation, "
        f"{'below' if chol_or.iloc[0] < 1 else 'above'} 1, which is {'the opposite of' if chol_or.iloc[0] < 1 else 'consistent with'} the textbook direction."
        if len(chol_or)
        else ""
    )
)

- **Age (years)** is the most used input: shuffling it lowers test ROC-AUC by 22.2 points, against 1.6 for the next, Total cholesterol (mg/dL).
- Using only age and sex gives 79.5%; all features give 82.9%; dropping every blood test gives 82.1%.
- The two models rank features similarly (Spearman 0.75).
- The logistic odds ratio for total cholesterol is 0.73 per standard deviation, below 1, which is the opposite of the textbook direction.

## What is weak

In [9]:
nb.say(
    "- **Importance is not causation.** If total cholesterol points in the opposite direction from the textbook (see the odds ratio above), one "
    "plausible reason is treatment: people with a diagnosis are often on cholesterol-lowering drugs. This is a hypothesis; the data here cannot test it. "
    "The model uses such patterns as predictors, which is fine for prediction on a similar survey and wrong as health advice.\n"
    "- SHAP and permutation importance split credit among correlated inputs (BMI, waist) arbitrarily, so single-feature rankings below the top are fragile.\n"
    f"- The SHAP plot explains XGBoost. The model selected for the demo is {R['selection']['selected'].replace('_', ' ')} (RESULTS.md section 5); "
    "for logistic regression the odds-ratio table is its explanation.\n"
    "- Nothing here shows that changing a feature (losing weight, quitting smoking) would change anyone's risk."
)

- **Importance is not causation.** If total cholesterol points in the opposite direction from the textbook (see the odds ratio above), one plausible reason is treatment: people with a diagnosis are often on cholesterol-lowering drugs. This is a hypothesis; the data here cannot test it. The model uses such patterns as predictors, which is fine for prediction on a similar survey and wrong as health advice.
- SHAP and permutation importance split credit among correlated inputs (BMI, waist) arbitrarily, so single-feature rankings below the top are fragile.
- The SHAP plot explains XGBoost. The model selected for the demo is logistic regression (RESULTS.md section 5); for logistic regression the odds-ratio table is its explanation.
- Nothing here shows that changing a feature (losing weight, quitting smoking) would change anyone's risk.